# SEA Pending Attrition / Transfer — Auto Reply
Scheduled daily: find → validate Expedia VN cases → check if replied → process + reply.

In [29]:
# ── CONFIG ────────────────────────────────────────────────────
SENDER_FILTER  = "SEA_WFM_ID_Deletion@concentrix.com"
SUBJECT_FILTER = "SEA - Client ID Deletion Notification"
ATTACH_KEYWORD = "SEA - Pending Attrition"
PROCESS_FILTER = "Expedia"
COUNTRY_FILTER = "Vietnam"
TARGET_FOLDER  = "GC3 + ExpAdmin"
OUR_EMAIL      = "huuchinh.nguyen@concentrix.com"

# Test recipients — replace with real addresses when ready
EMAIL_TO = "huuchinh.nguyen@concentrix.com;"
EMAIL_CC = "huuchinh.nguyen@concentrix.com;"

DISPLAY_PREVIEW  = True   # set False when running as scheduled .py
DAYS_LOOKBACK    = 7      # only check emails received within last N days

In [30]:
import os, re, time, pythoncom, win32com.client
import pandas as pd
from pathlib import Path
from datetime import datetime
from IPython.display import display, HTML

HOME     = os.path.expanduser("~").replace("\\", "/")
SAVE_DIR = Path(HOME) / "Downloads"
SAVE_DIR.mkdir(exist_ok=True)

def main():
    print(f"[{datetime.now():%Y-%m-%d %H:%M:%S}] Starting SEA Pending Attrition auto-reply")

    # ── Connect Outlook ───────────────────────────────────────────
    pythoncom.CoInitialize()
    ol = win32com.client.Dispatch("Outlook.Application")
    ns = ol.GetNamespace("MAPI")
    ns.Logon()

    # ── Folder helpers ────────────────────────────────────────────
    def find_named_folder(root_folders, name):
        for f in root_folders:
            try:
                if name.lower() in f.Name.lower(): return f
                for sub in f.Folders:
                    if name.lower() in sub.Name.lower(): return sub
            except: pass
        return None

    def search_folder(folder, sender_kw, subj_kw, recurse=True):
        kw_smtp   = sender_kw.lower()
        kw_name   = sender_kw.split("@")[0].lower().replace("_", " ")
        kw_nodash = sender_kw.split("@")[0].lower().replace("_", "")

        def sender_ok(item):
            sa = str(getattr(item, "SenderEmailAddress", "") or "").lower()
            sn = str(getattr(item, "SenderName",         "") or "").lower()
            return kw_smtp in sa or kw_name in sn or kw_nodash in sn.replace(" ", "")

        try:
            dasl = f"@SQL=\"urn:schemas:httpmail:subject\" LIKE '%{subj_kw}%'"
            from datetime import datetime as _dt, timedelta as _td, timezone as _tz
            _cutoff = (_dt.now(_tz.utc) - _td(days=DAYS_LOOKBACK)).strftime("%Y-%m-%d %H:%M")
            dasl += ' AND "urn:schemas:httpmail:datereceived" >= ' + repr(_cutoff)
            _res = folder.Items.Restrict(dasl)
            _res.Sort("[ReceivedTime]", True)  # newest first
            for item in _res:
                try:
                    if sender_ok(item): return item
                except: continue
        except:
            try:
                folder.Items.Sort("[ReceivedTime]", True)
                for i, item in enumerate(folder.Items):
                    if i >= 500: break
                    try:
                        if subj_kw.lower() in str(getattr(item, "Subject", "")).lower() and sender_ok(item):
                            return item
                    except: continue
            except: pass

        if recurse:
            try:
                for sub in folder.Folders:
                    found = search_folder(sub, sender_kw, subj_kw, recurse=True)
                    if found: return found
            except: pass
        return None

    # ── Find target email ─────────────────────────────────────────
    print(f"[Search] Sender: '{SENDER_FILTER}' | Subject: '{SUBJECT_FILTER}'")
    target_mail = None

    if TARGET_FOLDER.strip():
        tf = find_named_folder(ns.Folders, TARGET_FOLDER)
        if tf:
            print(f"[Search] Folder '{tf.Name}' ({tf.Items.Count} items)")
            target_mail = search_folder(tf, SENDER_FILTER, SUBJECT_FILTER)

    if target_mail is None:
        print("[Search] Fallback: scanning all mailbox folders")
        for root in ns.Folders:
            try:
                target_mail = search_folder(root, SENDER_FILTER, SUBJECT_FILTER)
                if target_mail: break
            except: continue

    if target_mail is None:
        print("[Skip] No matching email found. Done.")
        return

    print(f"[Found] {target_mail.Subject} | Received: {target_mail.ReceivedTime}")

    # ── Download attachment ───────────────────────────────────────
    print(f"[Attachments] {target_mail.Attachments.Count} total")
    raw_path = None
    for att in target_mail.Attachments:
        if ATTACH_KEYWORD.lower() in att.FileName.lower():
            raw_path = SAVE_DIR / att.FileName
            att.SaveAsFile(str(raw_path))
            print(f"[Downloaded] {raw_path.name}")
            break

    if raw_path is None:
        print(f"[Skip] Attachment '{ATTACH_KEYWORD}' not found. Done.")
        return

    # ── Read & validate: must have Expedia + Vietnam rows ─────────
    xl     = pd.ExcelFile(str(raw_path))
    sheets = xl.sheet_names
    print(f"[Excel] Sheets: {sheets}")

    def find_col(df, *keywords):
        for kw in keywords:
            match = next((c for c in df.columns if kw.lower() in c.lower()), None)
            if match: return match
        return None

    def filter_df(df, proc_col, country_col):
        mask = df[proc_col].str.strip().str.lower() == PROCESS_FILTER.lower()
        if country_col:
            mask &= df[country_col].str.strip().str.lower() == COUNTRY_FILTER.lower()
        return df[mask].copy()

    sheet_a = next((s for s in sheets if "attrition" in s.lower()), None)
    sheet_t = next((s for s in sheets if "transfer"  in s.lower()), None)

    df_a_raw = pd.read_excel(xl, sheet_name=sheet_a, dtype=str).fillna("") if sheet_a else pd.DataFrame()
    df_t_raw = pd.read_excel(xl, sheet_name=sheet_t, dtype=str).fillna("") if sheet_t else pd.DataFrame()
    xl.close()

    proc_col_a    = find_col(df_a_raw, "process")
    country_col_a = find_col(df_a_raw, "country")
    deact_col_a   = find_col(df_a_raw, "deactivated", "deleted")

    proc_col_t    = find_col(df_t_raw, "old process", "process")
    country_col_t = find_col(df_t_raw, "country")
    deact_col_t   = find_col(df_t_raw, "deactivated", "deleted")

    df_a = filter_df(df_a_raw, proc_col_a, country_col_a) if proc_col_a else pd.DataFrame()
    df_t = filter_df(df_t_raw, proc_col_t, country_col_t) if proc_col_t else pd.DataFrame()

    total_cases = len(df_a) + len(df_t)
    print(f"[Filter] Attrition: {len(df_a)} rows | Transfer: {len(df_t)} rows | Total: {total_cases}")

    if total_cases == 0:
        print(f"[Skip] No {PROCESS_FILTER} + {COUNTRY_FILTER} cases found. No reply needed.")
        return

    # ── Check if already replied by OUR_EMAIL ────────────────────
    def already_replied(email_item, sent_folder):
        """
        Check Sent Items for a reply to this email.
        Requires: match found AND sent AFTER original was received (time guard).
        """
        try:
            conv_id   = email_item.ConversationID
            orig_subj = email_item.Subject
            orig_recv = email_item.ReceivedTime
            subj_part = orig_subj[-20:] if len(orig_subj) > 20 else orig_subj
            dasl      = f"@SQL=\"urn:schemas:httpmail:subject\" LIKE '%{subj_part}%'"

            restricted = sent_folder.Items.Restrict(dasl)
            restricted.Sort("[SentOn]", True)

            for sent in restricted:
                try:
                    sent_subj = str(getattr(sent, "Subject", "") or "")
                    sent_conv = getattr(sent, "ConversationID", None)
                    sent_on   = getattr(sent, "SentOn", None)

                    conv_match = (sent_conv == conv_id)
                    subj_match = (sent_subj.lower().startswith("re:") and
                                  subj_part.lower() in sent_subj.lower())

                    if not (conv_match or subj_match):
                        continue

                    # Time guard: reply must be sent AFTER original email arrived
                    time_ok = False
                    if sent_on:
                        try:
                            s = sent_on.replace(tzinfo=None) if getattr(sent_on, "tzinfo", None) else sent_on
                            r = orig_recv.replace(tzinfo=None) if getattr(orig_recv, "tzinfo", None) else orig_recv
                            time_ok = s >= r
                        except Exception:
                            time_ok = True

                    print(f"  [Reply check] Found in Sent: {sent_subj!r} | SentOn: {sent_on} | TimeOK: {time_ok}")

                    if time_ok:
                        return True, sent_subj, sent_on

                except Exception:
                    continue

        except Exception as e:
            print(f"[Reply check] Error: {e}")
        return False, None, None
    sent_folder = ns.GetDefaultFolder(5)
    replied, reply_subj, reply_time = already_replied(target_mail, sent_folder)

    if replied:
        print(f"[Skip] Already replied: '{reply_subj}' at {reply_time}")
        return

    print("[Reply check] Not replied yet — proceeding")

    # ── Fill Y + save ─────────────────────────────────────────────
    if deact_col_a and len(df_a) > 0: df_a[deact_col_a] = "Y"
    if deact_col_t and len(df_t) > 0: df_t[deact_col_t] = "Y"

    safe_subj = re.sub(r'[\\/:*?"<>|]', "-", str(target_mail.Subject)).strip()
    out_path  = SAVE_DIR / f"Expedia VN - {safe_subj}.xlsx"

    with pd.ExcelWriter(str(out_path), engine="openpyxl", mode="w") as writer:
        if len(df_a) > 0: df_a.to_excel(writer, sheet_name=sheet_a or "Pending Attrition", index=False)
        if len(df_t) > 0: df_t.to_excel(writer, sheet_name=sheet_t or "Pending Transfer",  index=False)

    print(f"[Saved] {out_path.name}")

    # ── Build pivot HTML table ────────────────────────────────────
    def pivot_html(df, label, group_keywords):
        if df.empty: return ""
        gcols = [c for c in df.columns if any(k.lower() in c.lower() for k in group_keywords)]
        if not gcols: return f'<p style="color:#c00">No group columns for {label}</p>'

        pivot = df.groupby(gcols, dropna=False).size().reset_index(name="Count")
        total = int(pivot["Count"].sum())

        P  = "margin:0;padding:0;font-family:Calibri,Arial,sans-serif;font-size:11px;line-height:13px;mso-line-height-rule:exactly;"
        TH = "background:#1565C0;color:#fff;padding:2px 8px;border:1px solid #555;text-align:center;white-space:nowrap;"
        TD = "padding:0 8px;border:1px solid #ccc;text-align:center;white-space:nowrap;background:#fff;"
        TC = "padding:0 8px;border:1px solid #ccc;text-align:center;white-space:nowrap;background:#E3F2FD;font-weight:bold;"
        TT = "padding:0 8px;border:1px solid #555;text-align:center;white-space:nowrap;background:#1565C0;color:#fff;font-weight:bold;"

        hdr  = "<tr>" + "".join(f'<th style="{TH}"><p style="{P}color:#fff;font-weight:bold;">{c}</p></th>' for c in pivot.columns) + "</tr>"
        body = "".join(
            "<tr>" + "".join(
                f'<td style="{TC if c=="Count" else TD}"><p style="{P}">{v}</p></td>'
                for c, v in zip(pivot.columns, row)
            ) + "</tr>"
            for _, row in pivot.iterrows()
        )
        ncols = len(pivot.columns)
        tot_row = (
            "<tr>"
            + "".join(f'<td style="{TT}"><p style="{P}color:#fff;">{"Total" if i==0 else ""}</p></td>' for i in range(ncols-1))
            + f'<td style="{TT}"><p style="{P}color:#fff;">{total}</p></td></tr>'
        )
        return (
            f'<p style="font-weight:bold;font-family:Calibri,Arial,sans-serif;margin:16px 0 4px;font-size:13px;">{label}</p>'
            f'<table cellpadding="0" cellspacing="0" style="border-collapse:collapse;margin-bottom:14px;mso-table-lspace:0pt;mso-table-rspace:0pt;">'
            f'{hdr}{body}{tot_row}</table>'
        )

    GROUP_A = ["country", "location", "process", "job family", "lob"]
    GROUP_T = ["country", "location", "old process", "new process", "job family", "lob"]

    pivot_section = pivot_html(df_a, f"Pending Attrition — {PROCESS_FILTER} {COUNTRY_FILTER}", GROUP_A)
    pivot_section += pivot_html(df_t, f"Pending Transfer — {PROCESS_FILTER} {COUNTRY_FILTER}", GROUP_T)

    # ── Email body ────────────────────────────────────────────────
    FONT = "font-family:Calibri,Arial,sans-serif;"
    email_body = f"""
    <div style="{FONT}font-size:14px;color:#222;padding:16px 20px;">
      <p style="margin:0 0 10px;">Dear team,</p>
      <p style="margin:0 0 14px;">
        We have attached the file for accounts from
        <strong>{PROCESS_FILTER} {COUNTRY_FILTER}</strong>.<br>
        Please refer to the attached file.
      </p>
      {pivot_section}
      <p style="font-size:13px;color:#444;margin:16px 0 0;
                border-top:1px solid #e0e0e0;padding-top:12px;line-height:1.8;">
        Thanks &amp; Regards,<br>
        <strong>Chinh Nguyen</strong><br>
        Analyst, WFM Real Time Management<br>
        Level 4, Tower 1, OneHub Saigon, Lot C1-2, D1 Street,<br>
        Tan Phu Ward, District 9, Ho Chi Minh City, Vietnam<br>
        Ph: +84 986 473 419&nbsp;|&nbsp;
        <a href="mailto:huuchinh.nguyen@concentrix.com" style="color:#1155CC;">
          huuchinh.nguyen@concentrix.com</a>
      </p>
    </div>
    """

    # ── Preview (notebook only) ───────────────────────────────────
    if DISPLAY_PREVIEW:
        wrap = (
            "<!DOCTYPE html><html><head><meta charset='utf-8'>"
            "<style>body{margin:0;background:#e8e8e8;}.w{max-width:900px;margin:16px auto;"
            "background:#fff;border:1px solid #ccc;}</style></head>"
            "<body><div class='w'>" + email_body + "</div></body></html>"
        )
        esc = wrap.replace("&","&amp;").replace('"',"&quot;").replace("'","&#39;")
        display(HTML(f'<iframe srcdoc="{esc}" style="width:100%;border:1px solid #ddd;min-height:400px;" '
                     f'onload="this.style.height=(this.contentDocument.body.scrollHeight+30)+\'px\'"></iframe>'))

    # ── Send Reply-All (auto — no confirm) ───────────────────────
    print(f"[Send] Replying to: {target_mail.Subject}")
    print(f"       To: {EMAIL_TO} | CC: {EMAIL_CC}")
    print(f"       Attach: {out_path.name}")

    reply = target_mail.ReplyAll()
    reply.To = EMAIL_TO
    reply.CC = EMAIL_CC

    raw     = reply.HTMLBody
    q_match = re.search(r'<div\s+id=["\']divRplyFwdMsg["\']', raw, re.IGNORECASE)
    if q_match:
        quoted = raw[q_match.start():]
    else:
        hr_m   = re.search(r'<hr\s[^>]*>', raw, re.IGNORECASE)
        quoted = raw[hr_m.start():] if hr_m else ""

    reply.HTMLBody = email_body + (f"<div>{quoted}</div>" if quoted else "")
    reply.Attachments.Add(str(out_path.resolve()))
    reply.Send()
    time.sleep(2)

    print(f"[Done] Reply sent. Attrition: {len(df_a)} | Transfer: {len(df_t)}")
    print(f"[{datetime.now():%Y-%m-%d %H:%M:%S}] Finished")

main()


[2026-09-04 05:57:29] Starting SEA Pending Attrition auto-reply
[Search] Sender: 'SEA_WFM_ID_Deletion@concentrix.com' | Subject: 'SEA - Client ID Deletion Notification'
[Search] Folder 'GC3 + ExpAdmin' (180 items)
[Found] SEA - Client ID Deletion Notification (03-Sep-2026) | Received: 2026-09-03 11:15:01.363000+00:00
[Attachments] 1 total
[Downloaded] SEA - Pending Attrition- Transfer.xlsx
[Excel] Sheets: ['Pending Attrition', 'Pending Transfer', 'MailDBDLs']
[Filter] Attrition: 3 rows | Transfer: 0 rows | Total: 3
[Reply check] Not replied yet — proceeding
[Saved] Expedia VN - SEA - Client ID Deletion Notification (03-Sep-2026).xlsx


c:\Users\huuchinh.nguyen\AppData\Local\anaconda3\Lib\site-packages\IPython\core\display.py:431: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


Country,Location,Process,Job Family,Count
Vietnam,Ho Chi Minh,Expedia,Operations Group,3
Total,,,,3


[Send] Replying to: SEA - Client ID Deletion Notification (03-Sep-2026)
       To: huuchinh.nguyen@concentrix.com; | CC: huuchinh.nguyen@concentrix.com;
       Attach: Expedia VN - SEA - Client ID Deletion Notification (03-Sep-2026).xlsx
[Done] Reply sent. Attrition: 3 | Transfer: 0
[2026-09-04 05:57:33] Finished
